# Fused RMSNorm & Residual Add in Triton
Colab notebook for the GPU environment

_*Use a GPU runtime_

Clone the repository

In [ ]:
import os
import shutil

repo_url = "https://github.com/jarnesino/fused-rmsnorm-residual-add-triton.git"
repo_dir = "/content/fused-rmsnorm-residual-add-triton"

# Remove existing clone
%cd /content
if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)

# Clone again
!git clone "$repo_url" "$repo_dir"

Set up the environment

In [ ]:
%cd /content/fused-rmsnorm-residual-add-triton
!curl -LsSf https://astral.sh/uv/install.sh | sh
!pip install -q go-task-bin
!task gpu:sync
!task gpu:test

In [ ]:
os.environ["MPLBACKEND"] = "agg"

from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Image as ImageDisplay
from PIL import Image

Run tutorial benchmarks

In [ ]:
!task gpu:bench:tutorials

In [ ]:
ImageDisplay("benchmarks/results/softmax/softmax-bandwidth.png")

Run RMSNormResidualAdd benchmarks

In [ ]:
!task gpu:bench

In [ ]:
base = Path("benchmarks/results/tesla-t4")
latest = max(p for p in base.iterdir() if p.is_dir())

paths = [
    str(latest / f"forward_{axis}_{dtype}.png")
    for axis in ("N", "M")
    for dtype in ("bfloat16", "float16", "float32")
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, path in zip(axes.flat, paths, strict=True):
    ax.imshow(Image.open(path))
    ax.set_title(path.split("/")[-1].replace(".png", ""))
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
!cd benchmarks/results && zip -qr /content/results-tesla-t4.zip tesla-t4
from google.colab import files

files.download("/content/results-tesla-t4.zip")